# Feature Engineering and Feature Optimization for Machine Learning

This beginner-friendly notebook contains:

1. Feature Engineering
2. Creating new features
3. Date-time features
4. Interaction features
5. Encoding categorical features
6. Feature scaling
7. Feature selection
8. PCA dimensionality reduction
9. Random Forest model training
10. Model comparison

Dataset used: **Wine dataset from sklearn**


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


## 2. Load the Wine Dataset

In [ ]:
wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

df.head()


## 3. Explore the Dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())


# What is Feature Engineering?

Feature Engineering means creating new and useful features from existing data.

Example:

If we have:

`Age = 25`

We can create:

`Age Group = Young Adult`

This helps the machine learning model understand the data better.


## 4. Create New Features from Wine Dataset

We create two new features:

1. `alcohol_to_malic_acid`
2. `total_phenols_flavanoids`


In [ ]:
df['alcohol_to_malic_acid'] = df['alcohol'] / (df['malic_acid'] + 0.001)
df['total_phenols_flavanoids'] = df['total_phenols'] + df['flavanoids']

df[['alcohol', 'malic_acid', 'alcohol_to_malic_acid',
    'total_phenols', 'flavanoids', 'total_phenols_flavanoids']].head()


## 5. Simple BMI Example

BMI is a new feature created from weight and height.

Formula:

`BMI = Weight / Height²`


In [ ]:
students = pd.DataFrame({
    'Name': ['Asha', 'Ravi', 'Meena', 'John'],
    'Weight_kg': [40, 55, 70, 80],
    'Height_m': [1.45, 1.60, 1.75, 1.80]
})

students['BMI'] = students['Weight_kg'] / (students['Height_m'] ** 2)

students


## 6. Date-Time Feature Engineering

From one date, we can extract:

- Day
- Month
- Year
- Weekday


In [ ]:
sales = pd.DataFrame({
    'Purchase_Date': ['2026-06-15', '2026-06-16', '2026-12-25', '2026-01-01'],
    'Amount': [500, 800, 2500, 1000]
})

sales['Purchase_Date'] = pd.to_datetime(sales['Purchase_Date'])

sales['Day'] = sales['Purchase_Date'].dt.day
sales['Month'] = sales['Purchase_Date'].dt.month
sales['Year'] = sales['Purchase_Date'].dt.year
sales['Weekday'] = sales['Purchase_Date'].dt.day_name()

sales


## 7. Interaction Features

Interaction features combine two features.

Example:

`Income × Age`


In [ ]:
customers = pd.DataFrame({
    'Customer': ['A', 'B', 'C'],
    'Age': [25, 40, 60],
    'Income': [30000, 50000, 70000]
})

customers['Income_Age_Interaction'] = customers['Income'] * customers['Age']

customers


# Encoding Categorical Features

Machine learning algorithms require numerical inputs.

Text values such as Red, Blue, and Green must be converted into numbers.


## 8. Label Encoding

Label Encoding assigns a number to each category.

Example:

Red = 0  
Blue = 1  
Green = 2


In [ ]:
color_data = pd.DataFrame({
    'Color': ['Red', 'Blue', 'Green', 'Red', 'Blue']
})

label_encoder = LabelEncoder()
color_data['Color_Label_Encoded'] = label_encoder.fit_transform(color_data['Color'])

color_data


## 9. One-Hot Encoding

One-Hot Encoding creates a separate binary column for each category.


In [ ]:
one_hot_encoded = pd.get_dummies(color_data['Color'], prefix='Color')

one_hot_encoded


# Feature Scaling

Feature scaling brings numerical values to a similar range.

This is useful because some features may have large values and others may have small values.


## 10. Apply StandardScaler

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled_df.head()


# Feature Optimization

Feature Optimization means selecting or extracting the most useful features.

Benefits:

- Faster training
- Better accuracy
- Less noise
- Reduced overfitting


## 11. Correlation Checking

In [ ]:
correlation = df.corr(numeric_only=True)['target'].sort_values(ascending=False)

correlation


## 12. Feature Selection using SelectKBest

SelectKBest selects the best features based on statistical scores.


In [ ]:
selector = SelectKBest(score_func=f_classif, k=5)
X_selected = selector.fit_transform(X_scaled, y)

selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features)


# PCA: Principal Component Analysis

PCA reduces many features into fewer important components.

Simple explanation:

Imagine a school bag with many books.  
Some books are important, some are repeated, and some are not needed.

PCA keeps only the most important information.

Example:

`50 Features → 10 Principal Components`


## 13. Apply PCA

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2', 'PC3'])
pca_df['target'] = y

pca_df.head()


## 14. Explained Variance Ratio

This tells how much information each principal component keeps.


In [ ]:
print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

print("\nTotal Information Preserved:")
print(sum(pca.explained_variance_ratio_))


## 15. PCA Visualization

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(pca_df['PC1'], pca_df['PC2'], c=pca_df['target'])
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Visualization')
plt.show()


# Random Forest Model Training

We train three models:

1. Model using all features
2. Model using selected features
3. Model using PCA features


## 16. Model 1: Random Forest using All Features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

rf_all = RandomForestClassifier(random_state=42)
rf_all.fit(X_train, y_train)

y_pred_all = rf_all.predict(X_test)

print("Accuracy using all features:", accuracy_score(y_test, y_pred_all))
print(classification_report(y_test, y_pred_all))


## 17. Model 2: Random Forest using Selected Features

In [ ]:
X_selected_all = selector.fit_transform(X_scaled, y)

X_train_sel, X_test_sel, y_train_sel, y_test_sel = train_test_split(
    X_selected_all, y, test_size=0.2, random_state=42, stratify=y
)

rf_selected = RandomForestClassifier(random_state=42)
rf_selected.fit(X_train_sel, y_train_sel)

y_pred_selected = rf_selected.predict(X_test_sel)

print("Accuracy using selected features:", accuracy_score(y_test_sel, y_pred_selected))
print(classification_report(y_test_sel, y_pred_selected))


## 18. Model 3: Random Forest using PCA Features

In [ ]:
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

rf_pca = RandomForestClassifier(random_state=42)
rf_pca.fit(X_train_pca, y_train_pca)

y_pred_pca = rf_pca.predict(X_test_pca)

print("Accuracy using PCA features:", accuracy_score(y_test_pca, y_pred_pca))
print(classification_report(y_test_pca, y_pred_pca))


## 19. Compare Results

In [ ]:
results = pd.DataFrame({
    'Model': ['All Features', 'Selected Features', 'PCA Features'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_all),
        accuracy_score(y_test_sel, y_pred_selected),
        accuracy_score(y_test_pca, y_pred_pca)
    ]
})

results


In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(results['Model'], results['Accuracy'])
plt.xlabel('Model Type')
plt.ylabel('Accuracy')
plt.title('Comparison of Feature Optimization Techniques')
plt.ylim(0, 1)
plt.show()


# Final Summary

In this notebook, we learned:

- Feature Engineering creates useful new features.
- Encoding converts text categories into numbers.
- Scaling brings features to a common range.
- Feature Selection selects the best features.
- PCA reduces many features into fewer important components.
- Random Forest can be trained using all features, selected features, or PCA features.


# Practice Questions

1. Create one new feature using two columns from the Wine dataset.
2. Apply Label Encoding to a categorical column.
3. Apply One-Hot Encoding to a categorical column.
4. Scale the dataset using StandardScaler.
5. Select the best 5 features using SelectKBest.
6. Apply PCA with 2 components.
7. Train a Random Forest model using PCA features.
8. Compare accuracy before and after feature selection.
